# MODELO SARIMAX PARA EL CONTAMINANTE NO PARA BARCELONA

In [1]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
from pylab import rcParams
plt.style.use("fivethirtyeight")
plt.rcParams["lines.linewidth"] = 1.5
light_style = {
    "figure.facecolor": "#d9effb",   
    "axes.facecolor": "#d9effb",
    "savefig.facecolor": "#d9effb",
    "axes.grid": True,
    "axes.grid.which": "both",
    "axes.spines.left": True,
    "axes.spines.right": True,
    "axes.spines.top": True,
    "axes.spines.bottom": True,
    "grid.color": "#a9d3f2",
    "grid.linewidth": "0.8",
    "text.color": "#333333",
    "axes.labelcolor": "#333333",
    "axes.labelweight": "black",      
    "xtick.color": "#333333",
    "ytick.color": "#333333",
    "font.size": 12,
    "axes.titleweight": "bold",       
    "legend.fontsize": 12,
    "legend.title_fontsize": 12,
}
plt.rcParams.update(light_style)
rcParams["figure.figsize"] = (18, 7)

import sys
import importlib
from pathlib import Path

SCRIPTS_PATH = Path.cwd().parents[2]

if str(SCRIPTS_PATH) not in sys.path:
    sys.path.append(str(SCRIPTS_PATH))

import utils
importlib.reload(utils)
from utils import CARGA_Y_FILTRO, EVALUAR_METRICAS, BUSQUEDA_CONFIGURACIONES_SARIMAX, ENTRENAR_EVALUAR_SARIMAX

Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.


In [2]:
BASE_PATH = Path("..", "..", "..", "..")
FOLDER_DATA = BASE_PATH / "datasets" / "eda_archivos_cont_clima_indices"

## Carga de los datos y división del conjunto de datos

Cargamos los datos y filtramos las columnas que realmente necesitamos y como ciudad elegimos únicamente Barcelona.

In [3]:
df1 = CARGA_Y_FILTRO(
    contaminante="C6H6 (ug.m-3)",
    ciudad="Barcelona"
)

In [4]:
# ==============================================================================
# División cronológica del conjunto de datos
# ==============================================================================

train = df1[df1["Año"] <= 2020].copy()

validation = df1[
    (df1["Año"] >= 2021) &
    (df1["Año"] <= 2022)
].copy()

test = df1[df1["Año"] >= 2023].copy()

print(f"Entrenamiento: {train['Start'].min()} -> {train['Start'].max()}")
print(f"Validación:    {validation['Start'].min()} -> {validation['Start'].max()}")
print(f"Prueba:        {test['Start'].min()} -> {test['Start'].max()}")

print()
print(f"Nº muestras entrenamiento: {len(train):,}")
print(f"Nº muestras validación:    {len(validation):,}")
print(f"Nº muestras prueba:        {len(test):,}")

Entrenamiento: 2014-06-30 23:00:00 -> 2020-12-31 23:00:00
Validación:    2021-01-01 00:00:00 -> 2022-12-31 23:00:00
Prueba:        2023-01-01 00:00:00 -> 2024-12-31 23:00:00

Nº muestras entrenamiento: 56,986
Nº muestras validación:    17,520
Nº muestras prueba:        17,544


Para seguir con la forma en que Prophet denominaba a la variable objetivo y la columna temporal, renombramos las fechas por *ds* t la variable objetivo por *y*.

In [5]:
train = train.rename(columns={"Start": "ds", "C6H6 (ug.m-3)": "y"})
validation = validation.rename(columns={"Start": "ds", "C6H6 (ug.m-3)": "y"})
test = test.rename(columns={"Start": "ds", "C6H6 (ug.m-3)": "y"})

In [6]:
variables_exogenas= [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "surface_pressure",
    "cloudcover",
    "windspeed_10m",
    "shortwave_radiation",
    "boundary_layer_height",
    "NDVI",
    "NDBI"
]

Nos quedamos exclusivamente con las variables necesarias para el modelado.

In [7]:
# Lista de columnas que quieres mantener
columnas = ['ds', 'y'] + variables_exogenas

train = train[columnas]
validation = validation[columnas]
test = test[columnas]

# Eliminamos la primera fila del conjunto de entrenamiento
train = train.iloc[1:].reset_index(drop=True)

## Selección de los hiperparámetros

In [ ]:
resultados_sarimax, mejor_configuracion_sarimax, mejor_modelo_sarimax = (
    BUSQUEDA_CONFIGURACIONES_SARIMAX(
        train=train,
        validation=validation,
        variables_exogenas=variables_exogenas,
        periodo_estacional=24
    )
)


Mejores parámetros:

s = 24
p = 1
d = 1
q = 1
P = 4
D = 5
Q = 5

Criterio de Selección: AIC
MSE de validación: 0.169007


In [8]:
mejores_parametros = {
    "s": 24,
    "p": 1,
    "d": 1,
    "q": 1,
    "P": 4,
    "D": 5,
    "Q": 5
}

In [ ]:
modelo_sarimax, resultados_sarimax, predicciones_sarimax = (
    ENTRENAR_EVALUAR_SARIMAX(
        train=train,
        validation=validation,
        test=test,
        variables_exogenas=variables_exogenas,
        mejores_parametros=mejores_parametros
    )
)

Resultados de la evaluación del modelo
--------------------------------------
Error absoluto medio (MAE): 0.387106
Error cuadrático medio (MSE): 0.169007
Raíz del error cuadrático medio (RMSE): 0.411105
Error porcentual absoluto medio (MAPE): 42.68 %
Raíz del error cuadrático medio normalizada (NRMSE): 63.42 %
